In [ ]:
import cv2
import numpy as np
import easyocr
from ultralytics import YOLO
from datetime import datetime
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import os
import warnings
import torch
import logging

os.environ["YOLO_VERBOSE"] = "False"
warnings.filterwarnings("ignore")
logging.getLogger("ultralytics").setLevel(logging.ERROR)

SENDER_EMAIL    = "fa24-bse-118@students.cuisahiwal.edu.pk"
RECEIVER_EMAIL  = "mhasnain48776246@gmail.com"
EMAIL_PASSWORD  = "mmxf gqaf ddee aysu"

VIDEO_SOURCE = "Traffic.mp4"

FRAME_SKIP      = 2
RESIZE_WIDTH    = 800
RESIZE_HEIGHT   = 600
CONF_THRESHOLD  = 0.4

torch.set_num_threads(os.cpu_count())

VEHICLE_CLASSES = {2, 3, 5, 7}

COLOR_RANGES = {
    "Red"    : ([0,   120,  70], [10,  255, 255]),
    "Red2"   : ([160, 120,  70], [180, 255, 255]),
    "Blue"   : ([94,   80,   2], [126, 255, 255]),
    "Green"  : ([40,   40,  40], [70,  255, 255]),
    "Yellow" : ([20,  100, 100], [30,  255, 255]),
    "White"  : ([0,    0,  200], [180,  50, 255]),
    "Black"  : ([0,    0,   0],  [180,  255,  40]),
    "Silver" : ([0,    0,  150], [180,   30, 220]),
}

def detect_color(vehicle_img):
    if vehicle_img.size == 0:
        return "Unknown"

    hsv = cv2.cvtColor(vehicle_img, cv2.COLOR_BGR2HSV)
    max_pixels    = 0
    detected_color = "Unknown"
    red_pixels     = 0

    for color, (lower, upper) in COLOR_RANGES.items():
        mask   = cv2.inRange(hsv, np.array(lower), np.array(upper))
        pixels = cv2.countNonZero(mask)

        if color == "Red":
            red_pixels = pixels
            continue
        if color == "Red2":
            pixels += red_pixels
            color   = "Red"

        if pixels > max_pixels:
            max_pixels     = pixels
            detected_color = color

    return detected_color

def extract_plate_text(vehicle_img, reader):
    if vehicle_img.size == 0:
        return None

    gray  = cv2.cvtColor(vehicle_img, cv2.COLOR_BGR2GRAY)
    blur  = cv2.bilateralFilter(gray, 11, 17, 17)
    edged = cv2.Canny(blur, 30, 200)

    contours, _ = cv2.findContours(
        edged, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE
    )

    contours = sorted(contours, key=cv2.contourArea, reverse=True)[:15]

    for cnt in contours:
        perimeter = cv2.arcLength(cnt, True)
        approx    = cv2.approxPolyDP(cnt, 0.018 * perimeter, True)

        if len(approx) == 4:
            x, y, w, h = cv2.boundingRect(cnt)
            aspect_ratio = w / float(h)

            if w < 60 or h < 20 or not (1.5 <= aspect_ratio <= 6.0):
                continue

            plate_img = vehicle_img[y:y+h, x:x+w]

            if plate_img.size == 0:
                continue

            scale = max(1, 200 // h)
            if scale > 1:
                plate_img = cv2.resize(
                    plate_img, None,
                    fx=scale, fy=scale,
                    interpolation=cv2.INTER_CUBIC
                )

            plate_gray  = cv2.cvtColor(plate_img, cv2.COLOR_BGR2GRAY)
            _, plate_bin = cv2.threshold(
                plate_gray, 0, 255,
                cv2.THRESH_BINARY + cv2.THRESH_OTSU
            )

            ocr_results = reader.readtext(plate_bin)

            if ocr_results:
                best = max(ocr_results, key=lambda r: r[2])
                text, confidence = best[-2], best[-1]

                cleaned = "".join(c for c in text if c.isalnum()).upper()
                if confidence > 0.25 and len(cleaned) >= 4:
                    return cleaned

    return None

def send_email(color, plate):
    body = f"""\
🚨 Vehicle Detection Alert

Color : {color}
Plate : {plate}
Time  : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

-- Automated Vehicle Monitoring System --
"""
    msg = MIMEMultipart()
    msg["Subject"] = f"Vehicle Alert — Plate: {plate}"
    msg["From"]    = SENDER_EMAIL
    msg["To"]      = RECEIVER_EMAIL
    msg.attach(MIMEText(body, "plain"))

    try:
        with smtplib.SMTP("smtp.gmail.com", 587) as server:
            server.starttls()
            server.login(SENDER_EMAIL, EMAIL_PASSWORD)
            server.send_message(msg)
        print(f"[EMAIL SENT] Plate: {plate} | Color: {color}")
    except Exception as e:
        print(f"[EMAIL FAILED] {e}")

def draw_overlay(frame, x1, y1, x2, y2, color_name, plate_text):
    cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 100, 0), 2)

    label_color = f"Color: {color_name}"
    cv2.putText(
        frame, label_color,
        (x1, y2 + 22),
        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 80, 255), 2
    )

    if plate_text:
        cv2.putText(
            frame, f"Plate: {plate_text}",
            (x1, max(y1 - 10, 15)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 220, 0), 2
        )

print("[INFO] Loading YOLO model (CPU — Intel i7 optimised)...")
model = YOLO("yolov8n.pt")
model.to("cpu")

print("[INFO] Loading EasyOCR (English)...")
reader = easyocr.Reader(["en"], gpu=False)

print("[INFO] Opening video source...")
cap = cv2.VideoCapture(VIDEO_SOURCE)

if not cap.isOpened():
    raise FileNotFoundError(f"Cannot open video source: {VIDEO_SOURCE}")

detected_plates = set()
frame_count     = 0

print("[INFO] Starting Vehicle Monitoring System — press 'q' to quit.\n")

while True:
    ret, frame = cap.read()
    if not ret:
        print("[INFO] End of video stream.")
        break

    frame_count += 1

    if frame_count % FRAME_SKIP != 0:
        cv2.imshow("Vehicle Monitoring System", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
        continue

    frame = cv2.resize(frame, (RESIZE_WIDTH, RESIZE_HEIGHT))
    blur = cv2.GaussianBlur(frame, (3, 3), 0)

    results = model(blur, conf=CONF_THRESHOLD, verbose=False)

    for result in results:
        boxes   = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()

        for i, box in enumerate(boxes):
            class_id = int(classes[i])
            if class_id not in VEHICLE_CLASSES:
                continue

            x1, y1, x2, y2 = map(int, box[:4])

            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(RESIZE_WIDTH, x2), min(RESIZE_HEIGHT, y2)

            vehicle = frame[y1:y2, x1:x2]
            if vehicle.size == 0:
                continue

            vehicle_color = detect_color(vehicle)
            plate_text = extract_plate_text(vehicle, reader)

            if plate_text and plate_text not in detected_plates:
                send_email(vehicle_color, plate_text)
                detected_plates.add(plate_text)
                print(f"[DETECTED] Plate: {plate_text} | Color: {vehicle_color}")

            draw_overlay(frame, x1, y1, x2, y2, vehicle_color, plate_text)

    fps_text = f"Frame: {frame_count}"
    cv2.putText(
        frame, fps_text, (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (200, 200, 200), 2
    )

    cv2.imshow("Vehicle Monitoring System", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        print("[INFO] User quit.")
        break

cap.release()
cv2.destroyAllWindows()
print(f"\n[DONE] Total unique plates detected: {len(detected_plates)}")
print("Plates found:", detected_plates if detected_plates else "None")

Using CPU. Note: This module is much faster with a GPU.


[INFO] Loading YOLO model (CPU — Intel i7 optimised)...
[INFO] Loading EasyOCR (English)...
[INFO] Opening video source...
[INFO] Starting Vehicle Monitoring System — press 'q' to quit.

[EMAIL FAILED] (535, b'5.7.8 Username and Password not accepted. For more information, go to\n5.7.8  https://support.google.com/mail/?p=BadCredentials ffacd0b85a97d-46dcbac0c9dsm26864305f8f.19 - gsmtp')
[DETECTED] Plate: AP10AR0658 | Color: White
[EMAIL FAILED] (535, b'5.7.8 Username and Password not accepted. For more information, go to\n5.7.8  https://support.google.com/mail/?p=BadCredentials 5b1f17b1804b1-492690100c7sm190770045e9.12 - gsmtp')
[DETECTED] Plate: PAP10AR0658 | Color: Red
[EMAIL FAILED] (535, b'5.7.8 Username and Password not accepted. For more information, go to\n5.7.8  https://support.google.com/mail/?p=BadCredentials ffacd0b85a97d-46d86960983sm26014575f8f.4 - gsmtp')
[DETECTED] Plate: TSO9EC653I | Color: White
[EMAIL FAILED] (535, b'5.7.8 Username and Password not accepted. For more 